In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model inside a `while True` loop.

Each iteration:
1. Send the full message history to the model.
2. Check the response for any `function_call`.
3. If there is one, run the tool and append the tool output to `messages`.
4. If there are no function calls, `break` out of the loop.

So the stop condition is simple: **no function calls in the current response**.


In [2]:
%pip install opentelemetry-api opentelemetry-sdk


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /usr/local/python/3.12.1/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter(out=sys.stdout))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [5]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        # 1. Call the parent class constructor
        super().__init__(*args, **kwargs) 
        
    def rag_traced(self, query):

        
        with tracer.start_as_current_span("rag") as span:
            search_results = self.search_traced(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm_traced(prompt)
            span.set_attribute("rag", "rag")
            return response.output_text
            

    def search_traced(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            result=self.search(query, num_results)
            span.set_attribute("search", "search")
            return result

    def llm_traced(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = self.llm(prompt)
            span.set_attribute("llm_input_tokens", response.usage.input_tokens)
            span.set_attribute("llm_output_tokens", response.usage.output_tokens)
            span.set_attribute("llm_output_tokens", response.usage.cost)
            return response


In [6]:
from starter import index, client
rag_traced = RAGTraced(index=index, llm_client=client)


In [7]:
#Q1
q1_query = "How does the agentic loop keep calling the model until it stops?"
q1_answer = rag_traced.rag_traced(q1_query)
print(q1_answer)

AttributeError: 'ResponseUsage' object has no attribute 'cost'

In [3]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("llm_input_tokens"),
                    attrs.get("llm_output_tokens"),
                    attrs.get("llm_cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [4]:
import sys
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [15]:
import pandas as pd
conn = sqlite3.connect("traces.db")

df = pd.read_sql_query("SELECT * FROM spans", conn)
df.columns = ["span_name", "start_time", "end_time", "input_tokens", "output_tokens","cost"]

llm = df[df["span_name"] == "llm"]
print(llm["input_tokens"])
print("Unique values:", llm["input_tokens"].unique())

2    None
3    None
7    None
Name: input_tokens, dtype: object
Unique values: [None]


In [16]:
print(df)

  span_name           start_time             end_time input_tokens  \
0    search  1784940266301956189  1784940266304051889         None   
1    search  1784940266301956189  1784940266304051889         None   
2       llm  1784940266318492232  1784940267686503377         None   
3       llm  1784940266318492232  1784940267686503377         None   
4       rag  1784940266301918153  1784940267697637634         None   
5       rag  1784940266301918153  1784940267697637634         None   
6    search  1784940347923904427  1784940347927198300         None   
7       llm  1784940347932362126  1784940349315678656         None   
8       rag  1784940347923846171  1784940349319626468         None   

  output_tokens  cost  
0          None  None  
1          None  None  
2          None  None  
3          None  None  
4          None  None  
5          None  None  
6          None  None  
7          None  None  
8          None  None  
